# 💉 VaccineNLP · PhoBERT-v2 Multitask Classifier
## Phát hiện Tin giả Vắc-xin & Phân tích Thái độ Cộng đồng (Tiếng Việt)

**Tác giả:** Kim Mạnh Hưng (2211090016) · Đinh Lê Quỳnh Phương (2211090031)  
**Người hướng dẫn:** TS. Trần Lâm Quân  
**Trường:** Đại học Y tế Công cộng (HUPH)  
**Môi trường:** Kaggle · GPU T4 x1  

---

### Tổng quan kiến trúc

Notebook này huấn luyện **Động cơ Phân loại** (Discriminator) trong kiến trúc **Dual-Student Hybrid** của dự án VaccineNLP.
Mô hình sử dụng **PhoBERT-base-v2** (`vinai/phobert-base-v2`) — pretrained encoder tiếng Việt mạnh nhất hiện tại — 
kết hợp với **Multi-task Learning** để dự đoán đồng thời 3 trục phân loại:

| Task | Nhãn | Lý do quan trọng |
|---|---|---|
| **Misinformation** | Tin giả / Chính xác | Mục tiêu chính của đề tài |
| **Stance** | Ủng hộ / Phản đối / Trung lập | 3 nhãn chuẩn — Fallback Bucket cho edge cases |
| **Sentiment** | Tiêu cực / Trung tính / Tích cực | Hỗ trợ phân tích thái độ cộng đồng |

### Các quyết định thiết kế then chốt

- **Weighted CrossEntropyLoss**: Bắt buộc do mất cân bằng nhãn nghiêm trọng (class imbalance). Nhãn *Tin giả* chỉ chiếm ~15% corpus.
- **Multi-task Learning (3 heads)**: Chia sẻ encoder giúp regularization tự nhiên, tránh overfitting trên tập nhỏ.
- **Word Segmentation (underthesea)**: PhoBERT yêu cầu văn bản được tách từ tiếng Việt trước khi tokenize.
- **Early Stopping + OneCycleLR**: Tối ưu cho fine-tuning BERT trên GPU đơn.

```
Input Text (đã tách từ)
      ↓
  [PhoBERT Encoder - 110M params]
      ↓
  [CLS] pooler_output (768-dim)
      ↓           ↓           ↓
 Head: Misinfo  Head: Stance  Head: Sentiment
  (2 classes)   (3 classes)   (3 classes)
```

## ⚡ Quick Start — Cấu hình Trước Khi Chạy

> **Chỉ cần sửa 2 dòng dưới đây**, toàn bộ notebook sẽ chạy đúng.

### Bước 1 · Thêm Dataset vào Notebook
Trong Kaggle sidebar → **+ Add Data** → tìm dataset của bạn → Add.  
Sau khi add, đường dẫn sẽ có dạng: `/kaggle/input/<tên-dataset>/...`

### Bước 2 · Cập nhật Paths (Cell Section 3)
```python
TRAIN_PATH = "/kaggle/input/vaccinenlp-clean-data/05_model_ready/train_v2_seg_v3.jsonl"
TEST_PATH  = "/kaggle/input/vaccinenlp-clean-data/03_processed/benchmark_test_set_v3.jsonl"
```

### Bước 3 · Bật GPU
Kaggle sidebar → **Session options** → Accelerator → **GPU T4 x1** → Save.

### Bước 4 · Chạy All
**Run All** (Shift+Enter từng cell, hoặc menu **Run > Run All Cells**).

---

### Cấu trúc file dữ liệu cần có
```
/kaggle/input/<dataset>/
    ├── 05_model_ready/
    │   └── train_v2_seg_v3.jsonl      ← Training data (đã word-segment)
    └── 03_processed/
        └── benchmark_test_set_v3.jsonl ← Gold Test Set (186 mẫu)
```

### Cấu trúc mỗi dòng JSONL
```json
{"text_segmented": "vắc xin Pfizer an_toàn cho trẻ_em",
 "standardized_ids": [2, 0, 2]}
```
*(standardized_ids = [misinfo_id, stance_id, sentiment_id])*


## 1. Cài đặt Môi trường

Kaggle đã pre-install `torch` và `transformers` với CUDA — **không upgrade** để tránh conflict.  
Chỉ cần cài thêm `underthesea` cho word segmentation tiếng Việt.


In [ ]:
# Kaggle pre-installs torch + transformers với CUDA — KHÔNG upgrade
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

import transformers
print(f"Transformers: {transformers.__version__}")

# underthesea: word segmentation tiếng Việt (bắt buộc cho PhoBERT)
import subprocess; subprocess.run(["pip","install","-q","underthesea"])
print("✅ Setup hoàn tất!")


## 2. Import Thư viện & Cấu hình GPU

Kiểm tra GPU khả dụng. Notebook được thiết kế cho **Kaggle T4 GPU**.


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'
print(f"✅ Device: {DEVICE} | {gpu_name}")


## 4. Tải & Chuẩn bị Dữ liệu

### Nguồn dữ liệu
- **Train (90%)**: `train_v2_seg_v3.jsonl` — Silver Labels từ Gemma-4 31B (Teacher Model), đã được word-segment sẵn.
- **Test (Gold)**: `benchmark_test_set_v3.jsonl` — **186 mẫu** đã qua Human-in-the-Loop (HITL) validation, là Ground Truth duy nhất cho đánh giá cuối.

### Chiến lược Split
```
Train JSONL (1,670 mẫu)
    ├── 90% → Training Set  (~1,503 mẫu)
    └── 10% → Validation Set (~167 mẫu)  [stratify theo 'misinfo']

benchmark_test_set_v3.jsonl (186 mẫu) → Gold Test Set [KHÔNG chạm vào khi train]
```
Stratified split theo nhãn `misinfo` đảm bảo tỷ lệ *Tin giả* được bảo toàn trong Val set.


In [ ]:
# [CELL 3] Configuration & Data Loading (Taxonomy v3)
import os, torch, json, glob as _g
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# --- PARAMS ---
MODEL_NAME      = 'vinai/phobert-base-v2'
MAX_LEN         = 256
BATCH_SIZE      = 16
EPOCHS          = 10
LR              = 2e-5
SAVE_DIR        = '/kaggle/working/phobert_v2'

# --- Paths (v3) ---
_tr = _g.glob('/kaggle/input/**/train_v2_seg_v3.jsonl', recursive=True)
_te = _g.glob('/kaggle/input/**/benchmark_test_set_v3.jsonl', recursive=True)
TRAIN_PATH = _tr[0] if _tr else '/kaggle/input/vaccinenlp-clean-data/05_model_ready/train_v2_seg_v3.jsonl'
TEST_PATH  = _te[0] if _te else '/kaggle/input/vaccinenlp-clean-data/03_processed/benchmark_test_set_v3.jsonl'

# Taxonomy v3
N_CLASSES = {'misinfo': 2, 'stance': 3, 'sentiment': 3}
TASKS = ['misinfo', 'stance', 'sentiment']
HC_NAMES = {
    'misinfo':   ['Tin giả',   'Chính xác'],
    'stance':    ['Ủng hộ',    'Phản đối',   'Trung lập'],
    'sentiment': ['Tiêu cực',  'Trung tính',  'Tích cực'],
}
LABEL_NAMES = HC_NAMES

# --- Load & Split ---
print(f'📂 Loading: {TRAIN_PATH}')
df = pd.read_json(TRAIN_PATH, lines=True)
for i, t in enumerate(TASKS):
    df[t] = df['standardized_ids'].apply(lambda x: x[i])
df['text'] = df['text_segmented'] if 'text_segmented' in df.columns else df['text']

df_train, df_val = train_test_split(
    df, test_size=0.1, stratify=df['misinfo'], random_state=42
)

print(f'📂 Loading Test: {TEST_PATH}')
df_test = pd.read_json(TEST_PATH, lines=True)
for i, t in enumerate(TASKS):
    df_test[t] = df_test['standardized_ids'].apply(lambda x: x[i])
df_test['text'] = df_test['text_segmented'] if 'text_segmented' in df_test.columns else df_test['text']

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'✅ Config v3 loaded — Train:{len(df_train)} | Val:{len(df_val)} | Test:{len(df_test)}')
print(f'   SAVE_DIR: {SAVE_DIR}')
print(f'   HC_NAMES misinfo: {HC_NAMES["misinfo"]}')


## 5. PyTorch Dataset & DataLoader

`VaccineDataset` xử lý tokenization với PhoBERT tokenizer và đóng gói labels 3 task vào một batch.  
Padding về `MAX_LEN=256` để đảm bảo batch hợp lệ trên GPU.


In [ ]:
# [CELL 6] Dataset & DataLoader
from torch.utils.data import Dataset, DataLoader

class VaccineDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts  = df['text'].tolist()
        self.labels = {t: df[t].values for t in TASKS}
        self.tokenizer = tokenizer
        self.max_len   = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding='max_length', 
                             max_length=self.max_len, return_tensors='pt')
        out = {k: v.squeeze(0) for k, v in enc.items()}
        for t in TASKS:
            out[t] = torch.tensor(int(self.labels[t][idx]), dtype=torch.long)
        return out

train_loader = DataLoader(VaccineDataset(df_train, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(VaccineDataset(df_val, tokenizer, MAX_LEN),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(VaccineDataset(df_test, tokenizer, MAX_LEN),  batch_size=BATCH_SIZE)
print(f"✅ DataLoaders ready. Train batches: {len(train_loader)}")


## 6. Kiến trúc Mô hình: PhoBERT Multitask Classifier

### Thiết kế Multi-task Head
Ba classification head độc lập chia sẻ cùng một **PhoBERT encoder**.  
Lợi thế so với 3 mô hình đơn lẻ:
- **Regularization tự nhiên**: Gradient từ 3 task cùng cập nhật encoder, giảm overfitting.
- **Hiệu quả tính toán**: Chỉ 1 lần forward pass qua BERT thay vì 3 lần.
- **Transfer learning nội bộ**: Stance và Sentiment hỗ trợ gián tiếp Misinformation detection.

### Weighted CrossEntropyLoss — Lý do bắt buộc
Tập dữ liệu có mất cân bằng nhãn nghiêm trọng:  
- *Tin giả* chỉ ~15% → Model naive sẽ bias về *Chính xác*.  
- `class_weight='balanced'` tính trọng số tỷ lệ nghịch với tần suất nhãn.  
- Đây là **Hard Constraint** của dự án — không được thay bằng standard CrossEntropyLoss.


In [ ]:
# [CELL 10] Model Architecture & Loss
import torch.nn as nn
from transformers import AutoModel
from sklearn.utils.class_weight import compute_class_weight

class MultitaskEncoder(nn.Module):
    def __init__(self, model_name, n_classes):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size  = self.encoder.config.hidden_size
        self.heads   = nn.ModuleDict({
            t: nn.Linear(hidden_size, n_classes[t]) for t in TASKS
        })
        self.dropout = nn.Dropout(0.1)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.pooler_output if hasattr(out, 'pooler_output') else out.last_hidden_state[:,0,:]
        pooled = self.dropout(pooled)
        return {t: self.heads[t](pooled) for t in TASKS}

model = MultitaskEncoder(MODEL_NAME, N_CLASSES).to(DEVICE)

# Loss & Weights
TASK_WEIGHTS = [0.5, 0.3, 0.2]
w_m = compute_class_weight('balanced', classes=np.unique(df_train['misinfo']), y=df_train['misinfo'])
loss_fns = {
    'misinfo':   nn.CrossEntropyLoss(weight=torch.tensor(w_m, dtype=torch.float).to(DEVICE)),
    'stance':    nn.CrossEntropyLoss(),
    'sentiment': nn.CrossEntropyLoss()
}
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
print(f"✅ Model & Loss initialized on {DEVICE}")


## 7. Vòng Lặp Huấn luyện (Training Loop)

Hàm `run_epoch` dùng chung cho cả train và eval.  
**Early Stopping** dựa trên `val_loss` — giúp tự động chọn checkpoint tốt nhất mà không cần can thiệp thủ công.

**Loss tổng hợp** = `0.5 × L_misinfo + 0.3 × L_stance + 0.2 × L_sentiment`  
Trọng số ưu tiên Misinformation detection — mục tiêu chính của đề tài.


In [ ]:
# [CELL 11] Training Engine
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = {t:[] for t in TASKS}, {t:[] for t in TASKS}
    for batch in tqdm(loader, desc='Train' if train else 'Eval', leave=False):
        ids, mask = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE)
        if train: optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(ids, mask)
            loss = sum(TASK_WEIGHTS[i] * loss_fns[t](logits[t], batch[t].to(DEVICE)) for i, t in enumerate(TASKS))
            if train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item()
        for t in TASKS:
            all_labels[t].extend(batch[t].numpy())
            all_preds[t].extend(torch.argmax(logits[t], dim=1).cpu().numpy())
    avg_loss = total_loss / len(loader)
    f1s = {t: f1_score(all_labels[t], all_preds[t], average='macro',
                       labels=list(range(N_CLASSES[t])), zero_division=0) for t in TASKS}
    return avg_loss, f1s, all_preds, all_labels

history = {'train_loss':[], 'val_loss':[]}
for t in TASKS: history[f'f1_{t}'] = []

t0 = time.time()
best_f1       = 0.0
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    tr_loss, _, _, _     = run_epoch(model, train_loader, train=True)
    val_loss, val_f1s, _, _ = run_epoch(model, val_loader, train=False)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    for t in TASKS: history[f'f1_{t}'].append(val_f1s[t])
    avg_f1 = np.mean(list(val_f1s.values()))
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {tr_loss:.4f}/{val_loss:.4f} | Avg F1: {avg_f1:.4f}")
    if avg_f1 > best_f1:
        best_f1       = avg_f1
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_model.pt")
        print(f"  ⭐ New best model saved! (val_loss={val_loss:.4f})")

training_seconds = int(time.time() - t0)
print(f"\n✅ Training done in {training_seconds}s | Best avg F1: {best_f1:.4f}")


## 8. Trực quan hóa Quá trình Huấn luyện

Loss curves và Validation F1 theo epoch giúp chẩn đoán:
- **Overfitting**: Train loss giảm nhưng Val loss tăng
- **Underfitting**: Cả hai loss đều cao
- **Điểm dừng tối ưu**: Đường thẳng xanh lá (Best checkpoint)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PhoBERT-v2 · Training Curves', fontsize=14, fontweight='bold')

# ── Tính best_epoch theo AVG F1 (khớp với tiêu chí lưu model trong training loop) ──
avg_f1_history = [
    np.mean([history[f'f1_{t}'][e] for t in TASKS])
    for e in range(len(history['train_loss']))
]
best_ep = int(np.argmax(avg_f1_history))   # 0-indexed

# Loss curves
ax = axes[0]
epochs_range = range(len(history['train_loss']))
ax.plot(list(epochs_range), history['train_loss'], 'b-o', label='Train Loss', markersize=4)
ax.plot(list(epochs_range), history['val_loss'],   'r-s', label='Val Loss',   markersize=4)
ax.axvline(best_ep, color='green', ls='--', alpha=0.8,
           label=f'Best Epoch ({best_ep + 1})')  # +1 vì x-axis bắt đầu từ 0
ax.set_xticks(list(epochs_range))
ax.set_xticklabels([str(e + 1) for e in epochs_range])  # Hiển thị 1-indexed
ax.set(xlabel='Epoch', ylabel='Loss', title='Loss Curves')
ax.legend(); ax.grid(True, alpha=0.3)

# F1 curves
ax = axes[1]
colors = {'misinfo': '#e74c3c', 'stance': '#3498db', 'sentiment': '#2ecc71'}
for t, c in colors.items():
    ax.plot(list(epochs_range), history[f'f1_{t}'],
            '-o', color=c, label=t.capitalize(), markersize=4)
ax.axvline(best_ep, color='green', ls='--', alpha=0.8, label=f'Best Epoch ({best_ep + 1})')
ax.set_xticks(list(epochs_range))
ax.set_xticklabels([str(e + 1) for e in epochs_range])  # 1-indexed
ax.set(xlabel='Epoch', ylabel='Macro F1', title='Validation F1 (mỗi task)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {SAVE_DIR}/training_curves.png | Best epoch: {best_ep + 1} (Avg F1 = {avg_f1_history[best_ep]:.4f})")


## 9. Đánh giá trên Gold Test Set (Benchmark)

> ⚠️ **Tập này hoàn toàn tách biệt khỏi quá trình huấn luyện.**  
> 186 mẫu đã qua Human-in-the-Loop validation — đây là Ground Truth duy nhất cho so sánh mô hình.



In [ ]:
# [CELL 12] Final Evaluation (Taxonomy v3)
from sklearn.metrics import classification_report

model.load_state_dict(torch.load(f"{SAVE_DIR}/best_model.pt", map_location=DEVICE))
_, test_f1s, test_preds, test_labels = run_epoch(model, test_loader, train=False)

print("\n" + "="*50 + "\nBENCHMARK - GOLD TEST SET\n" + "="*50)
for t in TASKS:
    print(f"\nTASK: {t.upper()} (F1: {test_f1s[t]:.4f})")
    print(classification_report(test_labels[t], test_preds[t], labels=list(range(N_CLASSES[t])), target_names=HC_NAMES[t], zero_division=0))


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 📑 INTEROP CONTRACT — Schema kết quả đồng nhất giữa 3 model notebooks
# ════════════════════════════════════════════════════════════════════════
# Mỗi notebook model XUẤT 1 file JSON tên DUY NHẤT ở /kaggle/working/ (gốc):
#   PhoBERT → /kaggle/working/phobert_v2_results.json
#   XLM-R   → /kaggle/working/xlmr_v1_results.json
#   Gemma   → /kaggle/working/gemma_v3_results.json
#
# SCHEMA BẮT BUỘC (mọi notebook PHẢI tuân theo):
# {
#   "model":       str,                 # "phobert-v2" | "xlmr-base" | "gemma-4-4b-xai"
#   "timestamp":   str (ISO),
#   "misinfo":   {"macro_f1": float, "per_class": [f1_0, f1_1],          "support": [s0, s1]},
#   "stance":    {"macro_f1": float, "per_class": [f1_0, f1_1, f1_2],    "support": [s0, s1, s2]},
#   "sentiment": {"macro_f1": float, "per_class": [f1_0, f1_1, f1_2],    "support": [s0, s1, s2]}
# }
# Taxonomy v3: Misinfo(2) Stance(3) Sentiment(3)
# ════════════════════════════════════════════════════════════════════════

# [CELL 12b] Export — schema chuẩn INTEROP
import datetime, json, os

def _per_class_f1_support(y_true, y_pred, n_classes):
    from sklearn.metrics import f1_score as _f1
    f1s, sups = [], []
    for c in range(n_classes):
        yt = [1 if y==c else 0 for y in y_true]
        yp = [1 if y==c else 0 for y in y_pred]
        f1s.append(round(_f1(yt, yp, zero_division=0), 4))
        sups.append(int(sum(yt)))
    return f1s, sups

phobert_results = {
    'model':            'phobert-v2',
    'timestamp':        datetime.datetime.now().isoformat(),
    'val_loss_best':    round(float(best_val_loss), 4),
    'training_seconds': training_seconds,
}
for t in TASKS:
    f1s, sups = _per_class_f1_support(test_labels[t], test_preds[t], N_CLASSES[t])
    phobert_results[t] = {'macro_f1': round(test_f1s[t], 4),
                          'per_class': f1s, 'support': sups}

# Save BOTH to model dir (archive) AND /kaggle/working root (interop discovery)
os.makedirs(SAVE_DIR, exist_ok=True)
for _p in [f"{SAVE_DIR}/phobert_v2_results.json",
           "/kaggle/working/phobert_v2_results.json"]:
    with open(_p, 'w', encoding='utf-8') as f:
        json.dump(phobert_results, f, indent=2, ensure_ascii=False)

# Schema validation
for t in TASKS:
    assert len(phobert_results[t]['per_class']) == N_CLASSES[t], f'per_class len: {t}'
    assert len(phobert_results[t]['support'])   == N_CLASSES[t], f'support len: {t}'

print("✅ INTEROP JSON saved:")
print("   /kaggle/working/phobert_v2_results.json")
print(json.dumps(phobert_results, indent=2, ensure_ascii=False))


## 10. Xuất Kết quả — Định dạng đồng bộ với Gemma

File `benchmark_results.json` được xuất với cấu trúc **đồng bộ 100%** với file kết quả của Gemma-4,  
để đảm bảo script so sánh mô hình chạy được mà không cần chỉnh sửa.


## 11. Confusion Matrix — Phân tích Lỗi Trực quan

Confusion matrix trên Gold Test Set giúp xác định:
- Nhãn nào model hay nhầm lẫn nhất?
- False Negative quan trọng: Model bỏ sót *Tin giả* → phân loại thành *Chính xác*?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('PhoBERT-v2 · Confusion Matrix (Gold Test Set)', fontsize=14, fontweight='bold')

for i, t in enumerate(TASKS):
    present = sorted(set(test_labels[t]) | set(test_preds[t]))
    cm      = confusion_matrix(test_labels[t], test_preds[t], labels=present)
    names   = [LABEL_NAMES[t][j] for j in present if j < len(LABEL_NAMES[t])]
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        ax=axes[i], xticklabels=names, yticklabels=names
    )
    axes[i].set_title(f'{t.capitalize()}\nMacro F1 = {test_f1s[t]:.4f}', fontweight='bold')
    axes[i].set_ylabel('True Label')
    axes[i].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {SAVE_DIR}/confusion_matrices.png")


## 12. (Tùy chọn) Upload lên Hugging Face Hub

Model đã được public tại: [hung2903/phobert-vaccine-multitask](https://huggingface.co/hung2903/phobert-vaccine-multitask)


In [ ]:
# (Tùy chọn) Upload lên Hugging Face Hub
try:
    from huggingface_hub import login, HfApi
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

    login(token=HF_TOKEN)
    api = HfApi()
    api.upload_folder(
        folder_path=SAVE_DIR,
        repo_id="hung2903/phobert-vaccine-multitask",
        repo_type="model",
    )
    print("✅ Uploaded to HuggingFace!")
except Exception as e:
    print(f"⚠️  HF upload failed (non-fatal): {e}")

print("✅ Notebook hoàn tất!")
print(f"   Model weights : {SAVE_DIR}/best_model.pt")
print(f"   Results JSON  : {SAVE_DIR}/benchmark_results.json")
print(f"   Training plot : {SAVE_DIR}/training_curves.png")
print(f"   Confusion mat : {SAVE_DIR}/confusion_matrices.png")


In [ ]:
# [CELL 12] Final Evaluation (Taxonomy v3)
from sklearn.metrics import classification_report, f1_score

def run_evaluation(model, loader, title="Evaluation"):
    model.eval()
    all_labels = {t: [] for t in TASKS}
    all_preds  = {t: [] for t in TASKS}
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=title):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            
            outputs = model(input_ids, attention_mask)
            
            for t in TASKS:
                all_labels[t].extend(batch[t].numpy())
                all_preds[t].extend(torch.argmax(outputs[t], dim=1).cpu().numpy())
    
    print(f"\n=== {title} Report ===")
    for t in TASKS:
        print(f"\nTask: {t.upper()}")
        print(classification_report(all_labels[t], all_preds[t], 
                                    labels=list(range(N_CLASSES[t])),
                                    zero_division=0))
        f1 = f1_score(all_labels[t], all_preds[t], average='macro', 
                      labels=list(range(N_CLASSES[t])), zero_division=0)
        print(f"Macro F1: {f1:.4f}")

# Run on Test Set
run_evaluation(model, test_loader, title="Gold Test Set (v3)")
